# Task 3 Prototype Notebook  
## FairCredit Jurisdiction Navigator

**Project:** Jurisdiction-Aware AI Credit Decisioning Compliance Tool for American Express  
**Use case:** AI-supported credit card approval  
**Jurisdictions:** United States and European Union  
**Data:** Synthetic credit application data only. No real American Express customer data is used.

This notebook demonstrates the core logic of the prototype:

1. Load synthetic credit application data.
2. Train a simple credit approval model.
3. Exclude `synthetic_group_proxy` from model training.
4. Use `synthetic_group_proxy` only for post-model fairness monitoring.
5. Build a jurisdiction configuration layer for US and EU modes.
6. Generate US adverse action explanations.
7. Generate EU fairness and AI governance alerts.
8. Save key output tables and figures for the final report and presentation.


## 1. Prototype Scope

This notebook is not a production credit model. It is a simplified prototype designed to show the regulatory mechanics of a jurisdiction-aware RegTech tool.

The key idea is:

> The same AI-supported credit decision can have different compliance meanings under US and EU regulatory expectations.

- **US mode:** Focuses on individual adverse action explanations.
- **EU mode:** Focuses on high-risk AI governance, fairness monitoring, model drift, and human oversight.


In [ ]:
# Core packages
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine learning packages
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Output folder
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete.")


In [ ]:
# Load synthetic dataset
# Keep the CSV in the same GitHub repository folder as this notebook.
csv_path = Path("Task3_Synthetic_Data.csv")

# Fallback path for local testing in this ChatGPT workspace
if not csv_path.exists():
    csv_path = Path("/mnt/data/Task3_Synthetic_Data.csv")

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


In [ ]:
# Basic dataset checks
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum())

print("\nApproval decision distribution:")
display(df["approval_decision"].value_counts(normalize=True).rename("share").to_frame())

print("\nJurisdiction distribution:")
display(df["jurisdiction"].value_counts().to_frame())

print("\nSynthetic group proxy distribution:")
display(df["synthetic_group_proxy"].value_counts().to_frame())


## 2. Data Design

The dataset is synthetic and was created for demonstration purposes. It includes applicant-level variables that are plausible in a credit card application setting.

Important design choice:

`synthetic_group_proxy` is **not used to train the model**. It is only used for post-model fairness monitoring. This reflects the values audit in Task 2: removing sensitive group variables from training is not enough; the tool still needs to test whether outcomes differ across groups.


In [ ]:
# Define model target
# good_repayment_12m is the synthetic ground-truth label:
# 1 = applicant is simulated to repay well over 12 months
# 0 = applicant is simulated to perform poorly
target = "good_repayment_12m"

# Features used for model training
# IMPORTANT: synthetic_group_proxy is intentionally excluded from model training.
numeric_features = [
    "annual_income",
    "debt_to_income",
    "credit_history_years",
    "missed_payments_12m",
    "credit_utilization",
    "existing_customer",
    "customer_tenure_years",
    "spending_volatility",
    "requested_credit_limit"
]

categorical_features = [
    "employment_status",
    "age_group"
]

excluded_from_training = [
    "applicant_id",
    "jurisdiction",
    "region",
    "synthetic_group_proxy",
    "prob_good_repayment",
    "policy_score",
    "approval_decision"
]

X = df[numeric_features + categorical_features]
y = df[target]

print("Training features:")
print(numeric_features + categorical_features)

print("\nExcluded from training:")
print(excluded_from_training)


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y,
    df.index,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

# Logistic regression model
# Chosen because it is simple, transparent, and easier to explain in a regulatory prototype.
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

model.fit(X_train, y_train)

# Predictions
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.50).astype(int)

print("Model trained successfully.")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))
print("\nClassification report:")
print(classification_report(y_test, y_pred))


In [ ]:
# Create a scored dataset for the test sample
scored = df.loc[idx_test].copy()
scored["model_probability"] = y_proba
scored["model_prediction_good_repayment"] = y_pred

# Convert model probability into an operational credit decision
# This is not meant to reproduce real American Express policy.
# It is only used to demonstrate the compliance tool.
def assign_model_decision(prob):
    if prob >= 0.62:
        return "Approved"
    elif prob >= 0.48:
        return "Manual Review"
    else:
        return "Declined"

scored["model_decision"] = scored["model_probability"].apply(assign_model_decision)

print("Scored test sample created.")
display(scored[[
    "applicant_id",
    "jurisdiction",
    "annual_income",
    "debt_to_income",
    "credit_history_years",
    "missed_payments_12m",
    "credit_utilization",
    "employment_status",
    "synthetic_group_proxy",
    "model_probability",
    "model_decision"
]].head(10))

# Save scored output
scored.to_csv(OUTPUT_DIR / "scored_test_applications.csv", index=False)


## 3. Jurisdiction Configuration Layer

The tool uses a simple configuration layer. In a real system, this would be stored in a versioned rule engine with effective dates, legal owners, and audit logs.

For this prototype:

- US mode prioritises adverse action explanation.
- EU mode prioritises high-risk AI governance, fairness monitoring, drift monitoring, and human review.


In [ ]:
JURISDICTION_RULES = {
    "US": {
        "regulatory_focus": "Adverse action explainability and fair lending",
        "requires_adverse_action_reasons": True,
        "requires_fairness_dashboard": False,
        "requires_drift_monitoring": False,
        "human_review_threshold_low": 0.48,
        "human_review_threshold_high": 0.62,
        "notes": "US mode focuses on whether a declined applicant can receive specific and accurate reasons."
    },
    "EU": {
        "regulatory_focus": "High-risk AI governance and bias monitoring",
        "requires_adverse_action_reasons": True,
        "requires_fairness_dashboard": True,
        "requires_drift_monitoring": True,
        "approval_rate_disparity_threshold": 0.80,
        "psi_drift_threshold": 0.20,
        "human_review_threshold_low": 0.48,
        "human_review_threshold_high": 0.62,
        "notes": "EU mode treats creditworthiness AI as a high-risk AI system requiring broader monitoring and governance."
    }
}

pd.DataFrame(JURISDICTION_RULES).T


## 4. US Mode: Adverse Action Explanation

This section generates a simple adverse action explanation for declined or borderline applications.

The explanation is rule-based for transparency. It checks the applicant's risk factors and returns the most relevant reasons in plain language. In a more advanced version, SHAP or another explainability method could be added, but the prototype keeps the logic simple and auditable.


In [ ]:
def generate_adverse_action_reasons(row, max_reasons=3):
    """
    Generate plain-language adverse action reasons.
    This function is intentionally simple and auditable.
    """
    reasons = []

    if row["debt_to_income"] >= 0.45:
        reasons.append("High debt-to-income ratio")
    if row["credit_history_years"] < 3:
        reasons.append("Limited credit history")
    if row["missed_payments_12m"] >= 2:
        reasons.append("Recent missed payments")
    if row["credit_utilization"] >= 0.70:
        reasons.append("High credit utilisation")
    if row["spending_volatility"] >= 0.65:
        reasons.append("High spending volatility")
    if row["annual_income"] < 30000:
        reasons.append("Income level below internal risk threshold")
    if row["requested_credit_limit"] > row["annual_income"] * 0.20:
        reasons.append("Requested credit limit is high relative to income")
    if row["employment_status"] in ["Unemployed", "Student"]:
        reasons.append("Employment status indicates higher repayment uncertainty")

    if not reasons:
        reasons.append("Application falls below the model approval threshold")

    return reasons[:max_reasons]


def us_mode_output(applicant_row):
    """
    Produce a US-style compliance output for one applicant.
    """
    reasons = generate_adverse_action_reasons(applicant_row)

    output = {
        "jurisdiction": "US",
        "regulatory_focus": JURISDICTION_RULES["US"]["regulatory_focus"],
        "applicant_id": applicant_row["applicant_id"],
        "model_probability": round(applicant_row["model_probability"], 4),
        "model_decision": applicant_row["model_decision"],
        "adverse_action_reasons": reasons,
        "compliance_interpretation": (
            "If the application is declined, the institution should provide specific and accurate reasons. "
            "A generic explanation such as 'low model score' would not be sufficient."
        )
    }
    return output


# Select a declined or manual review applicant to demonstrate US mode
example_us = scored[scored["model_decision"].isin(["Declined", "Manual Review"])].iloc[0]
us_output = us_mode_output(example_us)

print("US Mode Output")
for key, value in us_output.items():
    print(f"{key}: {value}")


## 5. EU Mode: Fairness and High-Risk AI Monitoring

The EU mode evaluates the system at portfolio level. It uses `synthetic_group_proxy` only for monitoring, not for model training.

The key fairness metric in this prototype is approval-rate disparity:

> disparity ratio = lowest group approval rate / highest group approval rate

A ratio below 0.80 is flagged for human review in this prototype.


In [ ]:
def compute_fairness_metrics(data, decision_col="model_decision", group_col="synthetic_group_proxy"):
    """
    Compute group-level fairness metrics for approval outcomes and prediction errors.
    """
    temp = data.copy()
    temp["approved_binary"] = (temp[decision_col] == "Approved").astype(int)
    temp["predicted_good_binary"] = (temp["model_prediction_good_repayment"] == 1).astype(int)
    temp["actual_good_binary"] = temp["good_repayment_12m"].astype(int)

    group_summary = temp.groupby(group_col).agg(
        records=("applicant_id", "count"),
        approval_rate=("approved_binary", "mean"),
        avg_model_probability=("model_probability", "mean"),
        actual_good_rate=("actual_good_binary", "mean")
    ).reset_index()

    # Error metrics by group
    error_rows = []
    for group, g in temp.groupby(group_col):
        tn, fp, fn, tp = confusion_matrix(
            g["actual_good_binary"],
            g["predicted_good_binary"],
            labels=[0, 1]
        ).ravel()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan

        error_rows.append({
            group_col: group,
            "false_positive_rate": fpr,
            "false_negative_rate": fnr
        })

    error_summary = pd.DataFrame(error_rows)
    result = group_summary.merge(error_summary, on=group_col, how="left")

    max_approval = result["approval_rate"].max()
    min_approval = result["approval_rate"].min()
    disparity_ratio = min_approval / max_approval if max_approval > 0 else np.nan

    return result, disparity_ratio


fairness_table, approval_disparity_ratio = compute_fairness_metrics(scored)

print("Fairness monitoring table:")
display(fairness_table)

print("Approval-rate disparity ratio:", round(approval_disparity_ratio, 3))

# Save fairness table
fairness_table.to_csv(OUTPUT_DIR / "eu_fairness_monitoring_table.csv", index=False)


In [ ]:
def eu_mode_output(data):
    """
    Produce an EU-style portfolio-level compliance output.
    """
    fairness_table, disparity_ratio = compute_fairness_metrics(data)
    threshold = JURISDICTION_RULES["EU"]["approval_rate_disparity_threshold"]

    alert_triggered = disparity_ratio < threshold

    output = {
        "jurisdiction": "EU",
        "regulatory_focus": JURISDICTION_RULES["EU"]["regulatory_focus"],
        "system_classification": "High-risk AI system for creditworthiness assessment",
        "approval_rate_disparity_ratio": round(disparity_ratio, 3),
        "threshold": threshold,
        "fairness_alert_triggered": alert_triggered,
        "human_review_required": alert_triggered,
        "governance_interpretation": (
            "EU mode requires system-level monitoring, not only individual explanations. "
            "If fairness or drift thresholds are breached, the model should be reviewed by human compliance and model risk teams."
        )
    }

    return output, fairness_table


eu_output, eu_fairness_table = eu_mode_output(scored)

print("EU Mode Output")
for key, value in eu_output.items():
    print(f"{key}: {value}")


## 6. Drift Monitoring

This section simulates a shifted applicant population to test whether the model remains stable.

The prototype uses Population Stability Index (PSI) for selected variables. PSI is a common monitoring metric for distribution shift.

Interpretation used here:

- PSI below 0.10: limited shift
- PSI between 0.10 and 0.20: moderate shift
- PSI above 0.20: material shift requiring review


In [ ]:
def calculate_psi(expected, actual, buckets=10):
    """
    Calculate Population Stability Index for one numeric variable.
    expected: baseline distribution
    actual: shifted/current distribution
    """
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    quantiles = np.linspace(0, 1, buckets + 1)
    breakpoints = np.unique(np.quantile(expected, quantiles))

    # If too few unique breakpoints, return NaN
    if len(breakpoints) <= 2:
        return np.nan

    expected_counts = pd.cut(expected, bins=breakpoints, include_lowest=True).value_counts(normalize=True, sort=False)
    actual_counts = pd.cut(actual, bins=breakpoints, include_lowest=True).value_counts(normalize=True, sort=False)

    expected_pct = expected_counts.replace(0, 0.0001)
    actual_pct = actual_counts.replace(0, 0.0001)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


# Create a shifted version of the scored dataset to simulate economic stress or population change
shifted = scored.copy()
shifted["annual_income"] = shifted["annual_income"] * np.random.normal(0.92, 0.08, size=len(shifted))
shifted["debt_to_income"] = np.clip(shifted["debt_to_income"] + np.random.normal(0.08, 0.05, size=len(shifted)), 0.01, 0.99)
shifted["credit_utilization"] = np.clip(shifted["credit_utilization"] + np.random.normal(0.07, 0.06, size=len(shifted)), 0.01, 0.99)
shifted["missed_payments_12m"] = np.clip(shifted["missed_payments_12m"] + np.random.poisson(0.4, size=len(shifted)), 0, 8)

psi_variables = [
    "annual_income",
    "debt_to_income",
    "credit_utilization",
    "missed_payments_12m"
]

psi_rows = []
for var in psi_variables:
    psi_value = calculate_psi(scored[var], shifted[var])
    psi_rows.append({
        "variable": var,
        "psi": psi_value,
        "review_required": psi_value >= JURISDICTION_RULES["EU"]["psi_drift_threshold"]
    })

psi_table = pd.DataFrame(psi_rows)
display(psi_table)

# Save drift monitoring output
psi_table.to_csv(OUTPUT_DIR / "drift_monitoring_psi_table.csv", index=False)


## 7. Visual Outputs for Report and Slides

The next cells save simple charts that can be used in:

- Task3_Output_Screenshots.pdf
- Task3_Tool_Design_Report.pdf
- Task3_Senior_Management_Deck.pdf


In [ ]:
# Chart 1: Model decision distribution
decision_counts = scored["model_decision"].value_counts().reindex(["Approved", "Manual Review", "Declined"])

plt.figure(figsize=(7, 4))
decision_counts.plot(kind="bar")
plt.title("Model Decision Distribution")
plt.xlabel("Decision")
plt.ylabel("Number of Applications")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_decision_distribution.png", dpi=200)
plt.show()

# Chart 2: Approval rate by synthetic group proxy
approval_by_group = scored.assign(
    approved_binary=(scored["model_decision"] == "Approved").astype(int)
).groupby("synthetic_group_proxy")["approved_binary"].mean()

plt.figure(figsize=(7, 4))
approval_by_group.plot(kind="bar")
plt.title("Approval Rate by Synthetic Group Proxy")
plt.xlabel("Synthetic Group Proxy")
plt.ylabel("Approval Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "approval_rate_by_group.png", dpi=200)
plt.show()

# Chart 3: PSI drift table chart
plt.figure(figsize=(7, 4))
plt.bar(psi_table["variable"], psi_table["psi"])
plt.axhline(JURISDICTION_RULES["EU"]["psi_drift_threshold"], linestyle="--")
plt.title("Population Stability Index by Variable")
plt.xlabel("Variable")
plt.ylabel("PSI")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "psi_drift_monitoring.png", dpi=200)
plt.show()


## 8. Prototype Summary

This prototype demonstrates the core design of FairCredit Jurisdiction Navigator.

The same credit approval model is interpreted differently depending on the active jurisdiction:

- **US mode** produces individual-level adverse action explanations.
- **EU mode** produces system-level fairness monitoring, high-risk AI governance alerts, and drift checks.

The tool does not replace legal judgement or final credit approval decisions. It is a compliance and governance layer designed to support human review.


In [ ]:
# Export a compact summary table for the final report
prototype_summary = pd.DataFrame([
    {
        "component": "Synthetic data",
        "prototype_output": f"{len(df)} synthetic credit application records",
        "report_use": "Explains data source and limitations"
    },
    {
        "component": "Credit model",
        "prototype_output": "Logistic regression approval model",
        "report_use": "Demonstrates technical execution"
    },
    {
        "component": "US mode",
        "prototype_output": "Adverse action reasons for declined or borderline applicants",
        "report_use": "Shows individual-level explainability"
    },
    {
        "component": "EU mode",
        "prototype_output": "Fairness metrics and high-risk AI governance alert",
        "report_use": "Shows system-level monitoring"
    },
    {
        "component": "Drift monitoring",
        "prototype_output": "PSI table for shifted applicant population",
        "report_use": "Shows model governance and failure mode handling"
    }
])

display(prototype_summary)
prototype_summary.to_csv(OUTPUT_DIR / "prototype_summary.csv", index=False)

print("All key output files saved in the 'outputs' folder.")
print("Files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file)
